# RadiologyAI — Training & Exploration Notebook

This notebook covers:
1. Dataset exploration & statistics
2. Model architecture summary
3. Training loss / BLEU curves
4. Qualitative report generation samples
5. Grad-CAM visualisations
6. Full evaluation metrics

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import json, pickle

# Project imports
from ai_model.models.radiology_model import RadiologyReportModel, build_model
from datasets.scripts.prepare_dataset import Vocabulary
from datasets.scripts.dataloader import build_dataloaders
from ai_model.evaluation.metrics import evaluate_all, print_results
from ai_model.explainability.gradcam import GradCAMExplainer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

%matplotlib inline
plt.style.use('dark_background')

## 1. Dataset Exploration

In [ ]:
# Load annotations
ANN_PATH   = '../datasets/processed/annotations.json'
VOCAB_PATH = '../datasets/processed/vocab.pkl'

with open(ANN_PATH) as f:
    ann = json.load(f)

with open(VOCAB_PATH, 'rb') as f:
    vocab = pickle.load(f)

records = ann['records']
splits  = ann['splits']

print(f'Total records : {len(records)}')
print(f'Train / Val / Test : {len(splits["train"])} / {len(splits["val"])} / {len(splits["test"])}')
print(f'Vocabulary size     : {len(vocab)}')
print(f'Max sequence length : {ann["max_seq_len"]}')

In [ ]:
# Report length distribution
lengths = [len(r['report'].split()) for r in records]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lengths, bins=40, color='#63b3ed', edgecolor='none', alpha=0.85)
axes[0].axvline(np.mean(lengths), color='#f6ad55', lw=2, label=f'Mean = {np.mean(lengths):.1f}')
axes[0].axvline(np.median(lengths), color='#48bb78', lw=2, label=f'Median = {np.median(lengths):.1f}')
axes[0].set_title('Report Length Distribution (words)')
axes[0].set_xlabel('Words per report')
axes[0].legend()

# Top-20 most common words
from collections import Counter
word_counts = Counter()
for r in records:
    word_counts.update(r['report'].lower().split())

top20 = word_counts.most_common(20)
words, counts = zip(*top20)
axes[1].barh(list(reversed(words)), list(reversed(counts)), color='#63b3ed', alpha=0.85)
axes[1].set_title('Top-20 Most Frequent Words')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

print(f'\nReport length stats:')
print(f'  Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.1f}, Std: {np.std(lengths):.1f}')

In [ ]:
# Sample images & reports
import random
samples = random.sample(records, min(6, len(records)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, rec in zip(axes.flatten(), samples):
    try:
        img = Image.open(rec['image_path']).convert('RGB')
        ax.imshow(img, cmap='gray')
        ax.set_title(rec['report'][:80] + '...', fontsize=7, wrap=True)
    except Exception:
        ax.text(0.5, 0.5, 'Image\nnot found', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.suptitle('Sample Chest X-Rays with Reports', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Model Architecture

In [ ]:
import yaml
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

model = build_model(cfg['model'], vocab_size=len(vocab)).to(DEVICE)
total_params = model.count_parameters()
print(f'Total trainable parameters: {total_params:,}')
print(f'  ≈ {total_params/1e6:.1f}M')
print()

# Per-module breakdown
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f'  {name:<25} {params:>10,}  ({100*params/total_params:.1f}%)')

## 3. Training Curves

In [ ]:
# Load TensorBoard event logs and plot manually
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

tb_dir = '../logs/tensorboard'

try:
    ea = EventAccumulator(tb_dir)
    ea.Reload()

    def get_scalar(tag):
        events = ea.Scalars(tag)
        steps  = [e.step  for e in events]
        values = [e.value for e in events]
        return steps, values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    for tag, label, color in [
        ('Train/loss', 'Train Loss', '#63b3ed'),
        ('Val/loss',   'Val Loss',   '#f6ad55'),
    ]:
        try:
            s, v = get_scalar(tag)
            axes[0].plot(s, v, label=label, color=color, linewidth=2)
        except KeyError:
            pass

    axes[0].set_title('Training / Validation Loss')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.2)

    # BLEU-4
    try:
        s, v = get_scalar('Val/BLEU4')
        axes[1].plot(s, v, color='#48bb78', linewidth=2)
        axes[1].fill_between(s, v, alpha=0.15, color='#48bb78')
        axes[1].set_title('Validation BLEU-4')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BLEU-4')
        axes[1].grid(alpha=0.2)
    except KeyError:
        axes[1].text(0.5, 0.5, 'No BLEU data yet', ha='center', va='center',
                     transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'No TensorBoard logs found (run training first): {e}')
    # Simulate example curves for demonstration
    epochs = np.arange(1, 51)
    train_loss = 4.5 * np.exp(-0.08 * epochs) + 0.3 + np.random.randn(50)*0.05
    val_loss   = 4.8 * np.exp(-0.07 * epochs) + 0.5 + np.random.randn(50)*0.07
    bleu4      = 0.25 * (1 - np.exp(-0.1 * epochs)) + np.random.randn(50)*0.005

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, train_loss, color='#63b3ed', lw=2, label='Train')
    axes[0].plot(epochs, val_loss,   color='#f6ad55', lw=2, label='Val')
    axes[0].set_title('Loss Curves (simulated)')
    axes[0].legend(); axes[0].grid(alpha=0.2)

    axes[1].plot(epochs, bleu4, color='#48bb78', lw=2)
    axes[1].fill_between(epochs, bleu4, alpha=0.15, color='#48bb78')
    axes[1].set_title('BLEU-4 Curve (simulated)')
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

## 4. Qualitative Inference Samples

In [ ]:
from torchvision import transforms

# Load best model
ckpt_path = '../ai_model/checkpoints/best_model.pth'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'], strict=False)
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
else:
    print('No checkpoint found — using random weights (outputs will be nonsense)')

model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

test_indices = ann['splits']['test'][:5]
test_records = [records[i] for i in test_indices]

print('\n' + '='*70)
for rec in test_records:
    try:
        img = Image.open(rec['image_path']).convert('RGB')
        t   = transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            seqs, scores = model.generate(t, beam_size=5, max_len=120, min_len=20)
        gen_report = vocab.decode(seqs[0].cpu().tolist())
        gen_report = gen_report.strip().capitalize()
        if not gen_report.endswith('.'): gen_report += '.'

        print(f'\nUID: {rec["uid"]}')
        print(f'REFERENCE : {rec["report"][:120]}...')
        print(f'GENERATED : {gen_report[:120]}...')
        print(f'SCORE     : {scores[0].item():.3f}')
        print('-'*70)
    except Exception as e:
        print(f'Error on {rec["uid"]}: {e}')

## 5. Grad-CAM Visualisation

In [ ]:
explainer = GradCAMExplainer(model, image_size=224, device=DEVICE)

sample_rec = test_records[0]
try:
    orig_img = Image.open(sample_rec['image_path']).convert('RGB')
    heatmap, overlay = explainer.explain(orig_img)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(orig_img, cmap='gray')
    axes[0].set_title('Original X-Ray', fontsize=11)
    axes[0].axis('off')

    im = axes[1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title('Grad-CAM Heatmap', fontsize=11)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(overlay)
    axes[2].set_title('Overlay (α=0.5)', fontsize=11)
    axes[2].axis('off')

    plt.suptitle(f'Grad-CAM — {sample_rec["uid"]}', fontsize=13)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'Grad-CAM demo failed: {e}')

## 6. Full Evaluation Metrics

In [ ]:
from ai_model.evaluation.metrics import evaluate_all, print_results

# Run on a subset of test set
test_indices_eval = ann['splits']['test'][:100]
hyps, refs = [], []

for idx in test_indices_eval:
    rec = records[idx]
    try:
        img = Image.open(rec['image_path']).convert('RGB')
        t   = transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            seqs, _ = model.generate(t, beam_size=3, max_len=120, min_len=15)
        hyps.append(vocab.decode(seqs[0].cpu().tolist()))
        refs.append(rec['report'])
    except Exception:
        pass

if hyps:
    results = evaluate_all(hyps, refs, compute_met=True)
    print_results(results, title=f'Evaluation on {len(hyps)} test samples')

    # Bar chart of metrics
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#63b3ed','#4299e1','#3182ce','#2b6cb0','#f6ad55','#f6cc55','#48bb78','#68d391']
    keys   = list(results.keys())
    vals   = list(results.values())
    bars   = ax.bar(keys, vals, color=colors[:len(keys)], alpha=0.85, edgecolor='none')

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + .005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

    ax.set_ylim(0, 1.05)
    ax.set_title('Evaluation Metrics', fontsize=13)
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.2)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No samples evaluated — check dataset paths.')